# Stage 04 - Connect Real-Time Sources

Preserve the focused deterministic replay used to exercise governed during-test findings. It contains 15 raw records, 13 canonical events, five event types, and six source systems.

> Replay is runnable. Live mode must be explicitly configured and validated. The separate full-lifecycle stream contains 405 events across 14 topics.

In [ ]:
from pyspark.sql import functions as F
import json

replay_mode = True

source_files = {
    'events': 'Files/shared/integrated-test-data/projections/realtime/recorded_observed_events.jsonl',
    'rules': 'Files/shared/integrated-test-data/projections/realtime/finding_rules.json',
}

table_names = {
    'events': 'bronze_recorded_observed_events',
    'preservation': 'bronze_replay_preservation_metadata',
    'connection_modes': 'bronze_stream_connection_modes',
}

live_mode_placeholders = {
    'event_hubs_namespace': '<event-hubs-namespace>',
    'kafka_topic_or_event_hub': '<topic-or-event-hub>',
    'fabric_eventstream_item': '<fabric-eventstream-item>',
    'fabric_eventhouse_table_set': 'SensorObservation, SystemStatus, CommandIntegration, TestMarker',
}

for label, path in source_files.items():
    print(f'Shared release input: {label} -> {path}')

if not replay_mode:
    print('Configure the live Eventstream and Eventhouse path outside this notebook before continuing.')
    for name, value in live_mode_placeholders.items():
        print(f'{name}: {value}')


In [ ]:
def add_bronze_metadata(df, source_file):
    payload_columns = [F.col(column_name) for column_name in df.columns]
    return (
        df.withColumn('bronze_source_file', F.lit(source_file))
        .withColumn('bronze_transport_mode', F.lit('checked_in_replay'))
        .withColumn('bronze_loaded_at_utc', F.current_timestamp())
        .withColumn('bronze_record_sha256', F.sha2(F.to_json(F.struct(*payload_columns)), 256))
    )

rules_text = '\n'.join(row.value for row in spark.read.text(source_files['rules']).collect())
rules = json.loads(rules_text)

events_raw_df = spark.read.option('multiLine', 'false').json(source_files['events'])
bronze_events_df = add_bronze_metadata(events_raw_df, source_files['events'])

print(f"Ruleset loaded: {rules['ruleset_id']} version {rules['ruleset_version']}")
print(f'Replay events landed: {bronze_events_df.count()}')
print('Replay mode is the supported runnable path in this notebook.')


In [ ]:
event_type_counts = {
    row['event_type']: row['count']
    for row in bronze_events_df.groupBy('event_type').count().collect()
}

preservation_marker_rows = bronze_events_df.filter(F.col('event_type') == 'preservation.marker').select(
    'event_id',
    'marker_name',
    'marker_phase',
    'marker_status',
    'preserved_event_count',
    'preserved_source_count',
    'preserved_window_start_utc',
    'preserved_window_end_utc',
    'replay_manifest_id',
    'replay_digest',
).collect()

preservation_marker = preservation_marker_rows[0].asDict() if preservation_marker_rows else {}

bronze_preservation_df = spark.createDataFrame(
    [
        (
            rules['ruleset_id'],
            rules['ruleset_version'],
            source_files['events'],
            source_files['rules'],
            bronze_events_df.count(),
            len(event_type_counts),
            json.dumps(event_type_counts, sort_keys=True),
            preservation_marker.get('event_id'),
            preservation_marker.get('marker_name'),
            preservation_marker.get('marker_phase'),
            preservation_marker.get('marker_status'),
            preservation_marker.get('preserved_event_count'),
            preservation_marker.get('preserved_source_count'),
            preservation_marker.get('preserved_window_start_utc'),
            preservation_marker.get('preserved_window_end_utc'),
            preservation_marker.get('replay_manifest_id'),
            preservation_marker.get('replay_digest'),
        )
    ],
    [
        'ruleset_id',
        'ruleset_version',
        'events_source_file',
        'rules_source_file',
        'observed_row_count',
        'observed_event_type_count',
        'event_type_counts_json',
        'preservation_event_id',
        'preservation_marker_name',
        'preservation_marker_phase',
        'preservation_marker_status',
        'preserved_event_count',
        'preserved_source_count',
        'preserved_window_start_utc',
        'preserved_window_end_utc',
        'replay_manifest_id',
        'replay_digest',
    ],
).withColumn('metadata_loaded_at_utc', F.current_timestamp())

connection_mode_rows = [
    (
        'replay',
        source_files['events'],
        table_names['events'],
        'Upload the checked-in JSONL and finding rules JSON to the Lakehouse Files area, then run the notebooks in order.',
        'Deterministic repo-backed replay path. This is the supported runnable mode for the lesson.',
    ),
    (
        'live',
        '<event-hubs-or-kafka-source>',
        'Eventstream -> Eventhouse -> Lakehouse',
        'Create the Eventstream and Eventhouse items outside this notebook, then point them at the same replay-compatible envelope.',
        'Placeholder guidance only. This notebook does not provision or validate live connectors.',
    ),
]

bronze_connection_modes_df = spark.createDataFrame(
    connection_mode_rows,
    ['connection_mode', 'source_reference', 'target_reference', 'operator_action', 'notes'],
)


## Bronze review

Bronze stays source preserving. The duplicate row, the malformed row, and the preservation marker remain visible here so the downstream measures and findings can stay evidence-linked and replayable.


In [ ]:
for table_name, frame in {
    table_names['events']: bronze_events_df,
    table_names['preservation']: bronze_preservation_df,
    table_names['connection_modes']: bronze_connection_modes_df,
}.items():
    (
        frame.write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(table_name)
    )

spark.table(table_names['connection_modes']).show(truncate=False)
spark.table(table_names['preservation']).show(truncate=False)
spark.table(table_names['events']).select(
    'event_id',
    'event_type',
    'source_system',
    'source_instance_id',
    'sequence_number',
    'event_time_utc',
    'ingest_time_utc',
).orderBy('event_time_utc', 'ingest_time_utc').show(truncate=False)
